# Pipeline de Água — Projeto PI
Estágio, FEUP

## O que faz
Pipeline responsável por download, parsing, limpeza, deteção de anomalias, persistência em base de dados e envio de relatório por email de dados de contadores de água.

Funciona em vários ambientes: **Google Colab**, **Render** e **Flask local**.

**Para qualquer erro/sugestão deve entrar em contacto com um dos membros do Grupo PI.**

---

## Secrets necessários no Colab

### Obrigatório
| Secret | Exemplo | Descrição |
|---|---|---|
| `USER` | `PI` | Prefixo usado para os secrets do grupo |
| `PI_github_json` | `{"key": "ghp_..."}` | Token de acesso ao repo de dados no GitHub |

### Fonte de dados (escolher uma)
| Secret | Exemplo | Descrição |
|---|---|---|
| `REPO_OWNER` | `pedroccpimenta` | Owner do repo de dados |
| `REPO_NAME` | `datafiles` | Nome do repo de dados |
| `REPO_FOLDER` | `aqualog` | Pasta dentro do repo com os JSONs |
| `REPO_BRANCH` | `master` | Branch do repo de dados |

### Email (opcional)
| Secret | Exemplo | Descrição |
|---|---|---|
| `EMAIL_ENABLED` | `true` | Ativar envio de email |
| `EMAIL_TO` | `user@example.com` | Destinatário(s), separados por vírgula |
| `BREVO_API_KEY` | `xkeysib-...` | Chave API do Brevo (recomendado) |
| `BREVO_SENDER_NAME` | `PI Water` | Nome do remetente |
| `BREVO_SENDER_EMAIL` | `noreply@example.com` | Email do remetente |

### Bases de dados (opcional — ativar as que forem usar)
| Secret | Exemplo | Descrição |
|---|---|---|
| `MONGO_URI` | `mongodb+srv://...` | URI do MongoDB Atlas |
| `TIDB_HOST` | `gateway01.eu-central-1...` | Host do TiDB Cloud |
| `TIDB_USER` | `username.root` | Utilizador TiDB |
| `TIDB_PASSWORD` | `password` | Password TiDB |
| `TIDB_DATABASE` | `pi_water` | Base de dados TiDB |
| `CRATEDB_HOST` | `https://...cratedb.net:4200` | Host do CrateDB |
| `CRATEDB_USER` | `admin` | Utilizador CrateDB |
| `CRATEDB_PASSWORD` | `password` | Password CrateDB |

### Outras opções
| Secret | Default | Descrição |
|---|---|---|
| `SKIP_DB_WRITES` | `false` | Ignorar escrita nas BDs (útil para testes) |
| `DAYS_BACK` | `0` | Nº de dias para trás na descoberta de ficheiros |
| `ANOMALY_LOOKBACK_DAYS` | `2` | Dias de histórico para deteção de anomalias |
| `VERBOSE` | `true` | Logging detalhado |

---

Grupo PI, 2026


In [ ]:
from google.colab import userdata
import json
import os
import sys

user = userdata.get('USER')
token_json = userdata.get(f"{user}_github_json")
token = json.loads(token_json).get("key")

repo_name = "water_pipeline"
repo_url = f"https://{token}@github.com/HenriquePerry/water_pipeline.git"

!rm -rf {repo_name}
!git clone {repo_url}

path = f"/content/{repo_name}"
%cd $path
if path not in sys.path:
    sys.path.append(path)

%pip install -q -r requirements.txt

# Faz a pipeline usar o secret PI_github_json para ler os JSON do repo de dados
os.environ["GITHUB_SECRET_NAME"] = f"{user}_github_json"

import app

result = app.run_pipeline_now(runtime_label='google-colab')
print(json.dumps(result, indent=2, ensure_ascii=False, default=str))


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
